In [1]:
import xml.etree.ElementTree as ET
import csv
from collections import defaultdict
import re

def parse_cdr_xml(xml_file):
    """Parse CDR XML file and extract features."""
    
    # Register namespace if needed (the XML doesn't use namespaces)
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Data structures to store extracted information
    documents = []
    chemical_mentions = defaultdict(list)
    disease_mentions = defaultdict(list)
    relations = defaultdict(list)
    
    # Parse each document
    for doc in root.findall('document'):
        doc_id = doc.find('id').text
        
        # Get title and abstract text
        title_text = ""
        abstract_text = ""
        for passage in doc.findall('passage'):
            infon = passage.find('infon[@key="type"]')
            if infon is not None:
                if infon.text == 'title':
                    title_text = passage.find('text').text
                elif infon.text == 'abstract':
                    abstract_text = passage.find('text').text
        
        documents.append({
            'document_id': doc_id,
            'title': title_text or '',
            'abstract': abstract_text or ''
        })
        
        # Extract chemical annotations
        for annotation in doc.findall('.//annotation'):
            infon_type = annotation.find('infon[@key="type"]')
            infon_mesh = annotation.find('infon[@key="MESH"]')
            location = annotation.find('location')
            text_elem = annotation.find('text')
            
            if infon_type is not None and infon_mesh is not None:
                ann_type = infon_type.text
                mesh_id = infon_mesh.text
                
                if ann_type == 'Chemical':
                    chemical_mentions[doc_id].append({
                        'annotation_id': annotation.get('id'),
                        'mesh_id': mesh_id,
                        'text': text_elem.text if text_elem is not None else '',
                        'offset': location.get('offset') if location is not None else '',
                        'length': location.get('length') if location is not None else ''
                    })
                elif ann_type == 'Disease':
                    disease_mentions[doc_id].append({
                        'annotation_id': annotation.get('id'),
                        'mesh_id': mesh_id,
                        'text': text_elem.text if text_elem is not None else '',
                        'offset': location.get('offset') if location is not None else '',
                        'length': location.get('length') if location is not None else ''
                    })
        
        # Extract CID relations
        for relation in doc.findall('relation'):
            relation_type = relation.find('infon[@key="relation"]')
            chem_mesh = relation.find('infon[@key="Chemical"]')
            disease_mesh = relation.find('infon[@key="Disease"]')
            
            if relation_type is not None and chem_mesh is not None and disease_mesh is not None:
                relations[doc_id].append({
                    'relation_id': relation.get('id'),
                    'type': relation_type.text,
                    'chemical_mesh': chem_mesh.text,
                    'disease_mesh': disease_mesh.text
                })
    
    return documents, chemical_mentions, disease_mentions, relations

def create_document_summary_csv(documents, filename='cdr_document_summary.csv'):
    """Create CSV with document summaries."""
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['document_id', 'title', 'abstract_length', 'has_chemicals', 'has_diseases', 'has_relations']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for doc in documents:
            writer.writerow({
                'document_id': doc['document_id'],
                'title': doc['title'][:100] + '...' if len(doc['title']) > 100 else doc['title'],
                'abstract_length': len(doc['abstract'].split()) if doc['abstract'] else 0,
                'has_chemicals': 'Yes',
                'has_diseases': 'Yes',
                'has_relations': 'Yes'  # Simplified - you'd need to check actual data
            })

def create_chemical_annotations_csv(chemical_mentions, filename='cdr_chemical_annotations.csv'):
    """Create CSV with all chemical annotations."""
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['document_id', 'annotation_id', 'mesh_id', 'text', 'offset', 'length']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for doc_id, chems in chemical_mentions.items():
            for chem in chems:
                writer.writerow({
                    'document_id': doc_id,
                    'annotation_id': chem['annotation_id'],
                    'mesh_id': chem['mesh_id'],
                    'text': chem['text'],
                    'offset': chem['offset'],
                    'length': chem['length']
                })

def create_disease_annotations_csv(disease_mentions, filename='cdr_disease_annotations.csv'):
    """Create CSV with all disease annotations."""
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['document_id', 'annotation_id', 'mesh_id', 'text', 'offset', 'length']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for doc_id, diseases in disease_mentions.items():
            for disease in diseases:
                writer.writerow({
                    'document_id': doc_id,
                    'annotation_id': disease['annotation_id'],
                    'mesh_id': disease['mesh_id'],
                    'text': disease['text'],
                    'offset': disease['offset'],
                    'length': disease['length']
                })

def create_relations_csv(relations, filename='cdr_relations.csv'):
    """Create CSV with all CID relations."""
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['document_id', 'relation_id', 'relation_type', 'chemical_mesh', 'disease_mesh']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for doc_id, rels in relations.items():
            for rel in rels:
                writer.writerow({
                    'document_id': doc_id,
                    'relation_id': rel['relation_id'],
                    'relation_type': rel['type'],
                    'chemical_mesh': rel['chemical_mesh'],
                    'disease_mesh': rel['disease_mesh']
                })

def create_comprehensive_csv(documents, chemical_mentions, disease_mentions, relations, 
                            filename='cdr_comprehensive_data.csv'):
    """Create a comprehensive CSV with all information combined."""
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['document_id', 'title', 'abstract', 'chemical_mesh_ids', 
                     'disease_mesh_ids', 'relation_count', 'chemical_mentions_count', 
                     'disease_mentions_count']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for doc in documents:
            doc_id = doc['document_id']
            
            # Get unique chemical MESH IDs
            chem_mesh = set([c['mesh_id'] for c in chemical_mentions.get(doc_id, []) 
                           if c['mesh_id'] != '-1'])
            
            # Get unique disease MESH IDs
            dis_mesh = set([d['mesh_id'] for d in disease_mentions.get(doc_id, []) 
                          if d['mesh_id'] != '-1'])
            
            writer.writerow({
                'document_id': doc_id,
                'title': doc['title'],
                'abstract': doc['abstract'][:500] + '...' if len(doc['abstract']) > 500 else doc['abstract'],
                'chemical_mesh_ids': '|'.join(chem_mesh),
                'disease_mesh_ids': '|'.join(dis_mesh),
                'relation_count': len(relations.get(doc_id, [])),
                'chemical_mentions_count': len(chemical_mentions.get(doc_id, [])),
                'disease_mentions_count': len(disease_mentions.get(doc_id, []))
            })

def create_stats_csv(documents, chemical_mentions, disease_mentions, relations, 
                    filename='cdr_statistics.csv'):
    """Create a statistics CSV file."""
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Metric', 'Value'])
        
        # Calculate statistics
        total_docs = len(documents)
        total_chems = sum(len(v) for v in chemical_mentions.values())
        total_diseases = sum(len(v) for v in disease_mentions.values())
        total_relations = sum(len(v) for v in relations.values())
        
        # Count unique MESH IDs
        unique_chem_mesh = set()
        unique_dis_mesh = set()
        for chems in chemical_mentions.values():
            unique_chem_mesh.update([c['mesh_id'] for c in chems if c['mesh_id'] != '-1'])
        for diseases in disease_mentions.values():
            unique_dis_mesh.update([d['mesh_id'] for d in diseases if d['mesh_id'] != '-1'])
        
        writer.writerow(['Total Documents', total_docs])
        writer.writerow(['Total Chemical Mentions', total_chems])
        writer.writerow(['Total Disease Mentions', total_diseases])
        writer.writerow(['Total CID Relations', total_relations])
        writer.writerow(['Unique Chemical MESH IDs', len(unique_chem_mesh)])
        writer.writerow(['Unique Disease MESH IDs', len(unique_dis_mesh)])
        writer.writerow(['Average Chemicals per Document', round(total_chems/total_docs, 2)])
        writer.writerow(['Average Diseases per Document', round(total_diseases/total_docs, 2)])
        writer.writerow(['Average Relations per Document', round(total_relations/total_docs, 2)])

def main(xml_file):
    """Main function to parse XML and generate CSV files."""
    print(f"Parsing {xml_file}...")
    
    # Parse the XML file
    documents, chemical_mentions, disease_mentions, relations = parse_cdr_xml(xml_file)
    
    # Generate CSV files
    print("Generating CSV files...")
    create_document_summary_csv(documents)
    create_chemical_annotations_csv(chemical_mentions)
    create_disease_annotations_csv(disease_mentions)
    create_relations_csv(relations)
    create_comprehensive_csv(documents, chemical_mentions, disease_mentions, relations)
    create_stats_csv(documents, chemical_mentions, disease_mentions, relations)
    
    print("CSV files generated successfully:")
    print("  - cdr_document_summary.csv")
    print("  - cdr_chemical_annotations.csv")
    print("  - cdr_disease_annotations.csv")
    print("  - cdr_relations.csv")
    print("  - cdr_comprehensive_data.csv")
    print("  - cdr_statistics.csv")
    
    # Print some statistics
    print(f"\nStatistics:")
    print(f"  Total Documents: {len(documents)}")
    print(f"  Total Chemical Mentions: {sum(len(v) for v in chemical_mentions.values())}")
    print(f"  Total Disease Mentions: {sum(len(v) for v in disease_mentions.values())}")
    print(f"  Total CID Relations: {sum(len(v) for v in relations.values())}")

# Run the extraction
if __name__ == "__main__":
    xml_file = "CDR_TrainingSet.BioC.xml"  # Replace with your file path
    main(xml_file)

Parsing CDR_TrainingSet.BioC.xml...
Generating CSV files...
CSV files generated successfully:
  - cdr_document_summary.csv
  - cdr_chemical_annotations.csv
  - cdr_disease_annotations.csv
  - cdr_relations.csv
  - cdr_comprehensive_data.csv
  - cdr_statistics.csv

Statistics:
  Total Documents: 500
  Total Chemical Mentions: 5207
  Total Disease Mentions: 4363
  Total CID Relations: 1038


In [2]:
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict


def parse_cdr_xml(xml_file):
    """
    Parse CDR BioC XML file and extract:
    - Documents
    - Passages
    - Annotations (multi-location supported)
    - Relations (role-based extraction)
    """

    tree = ET.parse(xml_file)
    root = tree.getroot()

    documents_data = []
    passages_data = []
    annotations_data = []
    relations_data = []

    # -------------------------
    # Iterate over documents
    # -------------------------
    for document in root.findall("document"):

        doc_id_elem = document.find("id")
        doc_id = doc_id_elem.text if doc_id_elem is not None else None

        # -------------------------
        # Passages
        # -------------------------
        for p_idx, passage in enumerate(document.findall("passage")):

            passage_text_elem = passage.find("text")
            passage_text = passage_text_elem.text if passage_text_elem is not None else ""

            offset_elem = passage.find("offset")
            passage_offset = offset_elem.text if offset_elem is not None else "0"

            passage_type_elem = passage.find('infon[@key="type"]')
            passage_type = passage_type_elem.text if passage_type_elem is not None else None

            passages_data.append({
                "document_id": doc_id,
                "passage_id": p_idx,
                "passage_type": passage_type,
                "passage_offset": passage_offset,
                "passage_text": passage_text
            })

            # -------------------------
            # Annotations (multi-location safe)
            # -------------------------
            for annotation in passage.findall("annotation"):

                ann_id = annotation.attrib.get("id")

                text_elem = annotation.find("text")
                ann_text = text_elem.text if text_elem is not None else ""

                # Extract all infon metadata dynamically
                infon_data = {}
                for infon in annotation.findall("infon"):
                    key = infon.attrib.get("key")
                    value = infon.text
                    infon_data[key] = value

                # Multiple location support
                locations = annotation.findall("location")

                if not locations:
                    # If no location found, still preserve annotation
                    annotations_data.append({
                        "document_id": doc_id,
                        "passage_id": p_idx,
                        "annotation_id": ann_id,
                        "entity_text": ann_text,
                        "start_offset": None,
                        "length": None,
                        **infon_data
                    })
                else:
                    for loc in locations:
                        start = loc.attrib.get("offset")
                        length = loc.attrib.get("length")

                        annotations_data.append({
                            "document_id": doc_id,
                            "passage_id": p_idx,
                            "annotation_id": ann_id,
                            "entity_text": ann_text,
                            "start_offset": start,
                            "length": length,
                            **infon_data
                        })

        # -------------------------
        # Relations (CDR Correct)
        # -------------------------
        for relation in document.findall("relation"):

            rel_id = relation.attrib.get("id")

            # Extract infon metadata dynamically
            infon_data = {
                infon.attrib.get("key"): infon.text
                for infon in relation.findall("infon")
            }

            chemical_ref = None
            disease_ref = None

            for node in relation.findall("node"):
                role = node.attrib.get("role")
                refid = node.attrib.get("refid")

                if role == "Chemical":
                    chemical_ref = refid
                elif role == "Disease":
                    disease_ref = refid

            relations_data.append({
                "document_id": doc_id,
                "relation_id": rel_id,
                "chemical_annotation_id": chemical_ref,
                "disease_annotation_id": disease_ref,
                **infon_data
            })

        documents_data.append({
            "document_id": doc_id
        })

    # Convert to DataFrames
    df_documents = pd.DataFrame(documents_data)
    df_passages = pd.DataFrame(passages_data)
    df_annotations = pd.DataFrame(annotations_data)
    df_relations = pd.DataFrame(relations_data)

    return df_documents, df_passages, df_annotations, df_relations


# ---------------------------------------
# Export All CSV Files
# ---------------------------------------
def export_all_csv(df_documents, df_passages, df_annotations, df_relations):

    df_documents.to_csv("cdr_documents.csv", index=False)
    df_passages.to_csv("cdr_passages.csv", index=False)
    df_annotations.to_csv("cdr_annotations.csv", index=False)
    df_relations.to_csv("cdr_relations.csv", index=False)

    print("CSV files generated successfully:")
    print("  - cdr_documents.csv")
    print("  - cdr_passages.csv")
    print("  - cdr_annotations.csv")
    print("  - cdr_relations.csv")


# ---------------------------------------
# Statistics Function
# ---------------------------------------
def print_statistics(df_documents, df_annotations, df_relations):

    total_docs = len(df_documents)
    total_annotations = len(df_annotations)
    total_relations = len(df_relations)

    total_chemicals = len(df_annotations[df_annotations.get("type") == "Chemical"])
    total_diseases = len(df_annotations[df_annotations.get("type") == "Disease"])

    print("\nDataset Statistics")
    print("--------------------")
    print(f"Total Documents: {total_docs}")
    print(f"Total Annotations: {total_annotations}")
    print(f"  - Chemical Mentions: {total_chemicals}")
    print(f"  - Disease Mentions: {total_diseases}")
    print(f"Total Relations (CID): {total_relations}")


# ---------------------------------------
# Main Execution
# ---------------------------------------
if __name__ == "__main__":

    xml_file = "CDR_TrainingSet.BioC.xml"  # Change path if needed

    print(f"Parsing {xml_file} ...")

    df_documents, df_passages, df_annotations, df_relations = parse_cdr_xml(xml_file)

    export_all_csv(df_documents, df_passages, df_annotations, df_relations)

    print_statistics(df_documents, df_annotations, df_relations)

Parsing CDR_TrainingSet.BioC.xml ...
CSV files generated successfully:
  - cdr_documents.csv
  - cdr_passages.csv
  - cdr_annotations.csv
  - cdr_relations.csv

Dataset Statistics
--------------------
Total Documents: 500
Total Annotations: 9643
  - Chemical Mentions: 5209
  - Disease Mentions: 4434
Total Relations (CID): 1038
